In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()

while PROJECT_ROOT.name != "Fraud-detection-ML-V2" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
METRICS_DIR = PROJECT_ROOT / "outputs" / "metrics"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FEATURE_PATH = (
    PROCESSED_DATA_DIR
    / "ml_training_features.csv"
)

if not TRAIN_FEATURE_PATH.exists():
    raise FileNotFoundError(
        f"Training feature file not found:\n{TRAIN_FEATURE_PATH}"
    )

df = pd.read_csv(
    TRAIN_FEATURE_PATH
)

ML_FEATURES = [
    "amount",
    "amount_vs_avg_ratio",
    "txn_count_last_5min",
    "time_since_last_txn_sec",
    "distance_from_last_location_km",
    "merchant_category_is_new_for_user"
]

TARGET_COLUMN = "is_fraud"

required_columns = ML_FEATURES + [TARGET_COLUMN]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

X = df[ML_FEATURES].copy()
y = df[TARGET_COLUMN].astype(int).copy()

negative_count = int((y == 0).sum())
positive_count = int((y == 1).sum())

if positive_count == 0:
    raise ValueError(
        "No fraud samples found."
    )

scale_pos_weight = (
    negative_count / positive_count
)

print("========== PHASE 5 DATA LOADED ==========")
print("Dataset shape:", df.shape)
print("ML features:", ML_FEATURES)
print("Number of ML features:", len(ML_FEATURES))
print("Fraud count:", positive_count)
print("Legitimate count:", negative_count)
print("scale_pos_weight:", scale_pos_weight)
print("=========================================")

========== PHASE 5 DATA LOADED ==========
Dataset shape: (1296675, 9)
ML features: ['amount', 'amount_vs_avg_ratio', 'txn_count_last_5min', 'time_since_last_txn_sec', 'distance_from_last_location_km', 'merchant_category_is_new_for_user']
Number of ML features: 6
Fraud count: 7506
Legitimate count: 1289169
scale_pos_weight: 171.75179856115108


In [2]:
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("========== TUNING SPLIT ==========")
print("Training shape:", X_train.shape)
print("Validation shape:", X_valid.shape)
print()
print("Training fraud:", int((y_train == 1).sum()))
print("Training legitimate:", int((y_train == 0).sum()))
print()
print("Validation fraud:", int((y_valid == 1).sum()))
print("Validation legitimate:", int((y_valid == 0).sum()))
print("==================================")

========== TUNING SPLIT ==========
Training shape: (1037340, 6)
Validation shape: (259335, 6)

Training fraud: 6005
Training legitimate: 1031335

Validation fraud: 1501
Validation legitimate: 257834


In [3]:
from scipy.stats import randint, uniform, loguniform

parameter_distributions = {
    "n_estimators": randint(
        200,
        600
    ),
    "max_depth": randint(
        3,
        10
    ),
    "learning_rate": loguniform(
        0.02,
        0.20
    ),
    "min_child_weight": randint(
        1,
        10
    ),
    "subsample": uniform(
        0.70,
        0.30
    ),
    "colsample_bytree": uniform(
        0.70,
        0.30
    ),
    "gamma": uniform(
        0.0,
        2.0
    ),
    "reg_alpha": loguniform(
        1e-4,
        1.0
    ),
    "reg_lambda": loguniform(
        0.1,
        10.0
    )
}

print("========== PARAMETER SEARCH SPACE ==========")

for name, distribution in parameter_distributions.items():
    print(
        f"{name}:",
        distribution
    )

print("============================================")

========== PARAMETER SEARCH SPACE ==========
n_estimators: <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x0000026EF7D64710>
max_depth: <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x0000026EFF738750>
learning_rate: <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x0000026EFFF964D0>
min_child_weight: <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x0000026EFFF96550>
subsample: <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x0000026EFFF97050>
colsample_bytree: <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x0000026EFFF97610>
gamma: <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x0000026EFFF97C10>
reg_alpha: <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x0000026EFFF982D0>
reg_lambda: <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x0000026EFFF98890>


In [4]:
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV

search_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    scale_pos_weight=scale_pos_weight,
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

random_search = RandomizedSearchCV(
    estimator=search_model,
    param_distributions=parameter_distributions,
    n_iter=15,
    scoring="average_precision",
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=1,
    refit=True
)

random_search.fit(
    X_train,
    y_train
)

print("========== HYPERPARAMETER SEARCH ==========")
print("Search completed:", True)
print("Configurations tested:", 15)
print("Best validation PR-AUC:", random_search.best_score_)
print("===========================================")

Fitting 3 folds for each of 15 candidates, totalling 45 fits
========== HYPERPARAMETER SEARCH ==========
Search completed: True
Configurations tested: 15
Best validation PR-AUC: 0.4654427179137611


In [5]:
best_parameters = random_search.best_params_

print("========== BEST XGBOOST PARAMETERS ==========")

for parameter, value in best_parameters.items():
    print(
        f"{parameter}:",
        value
    )

print("============================================")

========== BEST XGBOOST PARAMETERS ==========
colsample_bytree: 0.9486212527455787
gamma: 0.7135066533871786
learning_rate: 0.03819130560209075
max_depth: 6
min_child_weight: 9
n_estimators: 356
reg_alpha: 0.16172900811143134
reg_lambda: 0.14096175149815865
subsample: 0.9960660809801551


In [6]:
best_model = random_search.best_estimator_

tuned_valid_probability = (
    best_model.predict_proba(
        X_valid
    )[:, 1]
)

tuned_valid_prediction = (
    tuned_valid_probability >= 0.5
).astype(int)

print("========== TUNED PREDICTIONS ==========")
print(
    "Validation predictions:",
    len(tuned_valid_prediction)
)

print(
    "Minimum probability:",
    float(tuned_valid_probability.min())
)

print(
    "Maximum probability:",
    float(tuned_valid_probability.max())
)

print("=======================================")

========== TUNED PREDICTIONS ==========
Validation predictions: 259335
Minimum probability: 0.0001646991295274347
Maximum probability: 0.9996539354324341


In [7]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

tuned_accuracy = accuracy_score(
    y_valid,
    tuned_valid_prediction
)

tuned_precision = precision_score(
    y_valid,
    tuned_valid_prediction,
    zero_division=0
)

tuned_recall = recall_score(
    y_valid,
    tuned_valid_prediction,
    zero_division=0
)

tuned_f1 = f1_score(
    y_valid,
    tuned_valid_prediction,
    zero_division=0
)

tuned_roc_auc = roc_auc_score(
    y_valid,
    tuned_valid_probability
)

tuned_pr_auc = average_precision_score(
    y_valid,
    tuned_valid_probability
)

tuned_confusion = confusion_matrix(
    y_valid,
    tuned_valid_prediction
)

print("========== TUNED MODEL METRICS ==========")

print(
    "Accuracy:",
    tuned_accuracy
)

print(
    "Precision:",
    tuned_precision
)

print(
    "Recall:",
    tuned_recall
)

print(
    "F1:",
    tuned_f1
)

print(
    "ROC-AUC:",
    tuned_roc_auc
)

print(
    "PR-AUC:",
    tuned_pr_auc
)

print()
print("Confusion Matrix:")
print(tuned_confusion)

print()
print("Classification Report:")
print(
    classification_report(
        y_valid,
        tuned_valid_prediction,
        zero_division=0
    )
)

print("=========================================")

========== TUNED MODEL METRICS ==========
Accuracy: 0.9320068637091021
Precision: 0.07036326834984553
Recall: 0.8800799467021986
F1: 0.130308261405672
ROC-AUC: 0.9723830722685778
PR-AUC: 0.4777052814578071

Confusion Matrix:
[[240381  17453]
 [   180   1321]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.93      0.96    257834
           1       0.07      0.88      0.13      1501

    accuracy                           0.93    259335
   macro avg       0.53      0.91      0.55    259335
weighted avg       0.99      0.93      0.96    259335



In [8]:
BASELINE_METRICS_PATH = (
    METRICS_DIR
    / "xgboost_baseline_metrics.json"
)

if not BASELINE_METRICS_PATH.exists():
    raise FileNotFoundError(
        f"Baseline metrics not found:\n{BASELINE_METRICS_PATH}"
    )

with open(
    BASELINE_METRICS_PATH,
    "r"
) as file:
    baseline_metrics = json.load(file)

comparison = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC",
        "PR-AUC"
    ],
    "Baseline": [
        baseline_metrics["accuracy"],
        baseline_metrics["precision"],
        baseline_metrics["recall"],
        baseline_metrics["f1"],
        baseline_metrics["roc_auc"],
        baseline_metrics["pr_auc"]
    ],
    "Tuned": [
        tuned_accuracy,
        tuned_precision,
        tuned_recall,
        tuned_f1,
        tuned_roc_auc,
        tuned_pr_auc
    ]
})

comparison["Change"] = (
    comparison["Tuned"]
    - comparison["Baseline"]
)

print("========== BASELINE VS TUNED ==========")
print(comparison.to_string(index=False))
print("=======================================")

========== BASELINE VS TUNED ==========
   Metric  Baseline    Tuned    Change
 Accuracy  0.933977 0.932007 -0.001970
Precision  0.072145 0.070363 -0.001781
   Recall  0.877415 0.880080  0.002665
       F1  0.133327 0.130308 -0.003018
  ROC-AUC  0.972398 0.972383 -0.000015
   PR-AUC  0.449583 0.477705  0.028122


In [9]:
TUNED_MODEL_PATH = (
    MODELS_DIR
    / "xgboost_tuned.json"
)

best_model.save_model(
    TUNED_MODEL_PATH
)

print("========== TUNED MODEL SAVED ==========")
print(
    "Model path:",
    TUNED_MODEL_PATH
)

print(
    "Model exists:",
    TUNED_MODEL_PATH.exists()
)

print("=======================================")

========== TUNED MODEL SAVED ==========
Model path: C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2\models\xgboost_tuned.json
Model exists: True


In [10]:
tuning_results = {
    "model": "XGBoost",
    "feature_count": len(ML_FEATURES),
    "features": ML_FEATURES,
    "best_cv_pr_auc": float(
        random_search.best_score_
    ),
    "best_parameters": {
        key: (
            float(value)
            if isinstance(value, (np.floating, float))
            else int(value)
            if isinstance(value, (np.integer, int))
            else value
        )
        for key, value in best_parameters.items()
    },
    "validation_accuracy": float(tuned_accuracy),
    "validation_precision": float(tuned_precision),
    "validation_recall": float(tuned_recall),
    "validation_f1": float(tuned_f1),
    "validation_roc_auc": float(tuned_roc_auc),
    "validation_pr_auc": float(tuned_pr_auc),
    "threshold": 0.5,
    "scale_pos_weight": float(scale_pos_weight)
}

TUNING_RESULTS_PATH = (
    METRICS_DIR
    / "xgboost_tuning_results.json"
)

with open(
    TUNING_RESULTS_PATH,
    "w"
) as file:
    json.dump(
        tuning_results,
        file,
        indent=4
    )

print("========== TUNING RESULTS SAVED ==========")
print(
    "Results file:",
    TUNING_RESULTS_PATH
)

print(
    "Results file exists:",
    TUNING_RESULTS_PATH.exists()
)

print("==========================================")

========== TUNING RESULTS SAVED ==========
Results file: C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2\outputs\metrics\xgboost_tuning_results.json
Results file exists: True


In [11]:
reloaded_tuned_model = XGBClassifier()

reloaded_tuned_model.load_model(
    TUNED_MODEL_PATH
)

reload_probability = (
    reloaded_tuned_model
    .predict_proba(
        X_valid.head(20)
    )[:, 1]
)

reload_valid = (
    np.isfinite(
        reload_probability
    ).all()
    and
    (
        (reload_probability >= 0)
        &
        (reload_probability <= 1)
    ).all()
)

print("========== TUNED MODEL RELOAD ==========")
print(
    "Reload successful:",
    True
)

print(
    "Probability output valid:",
    reload_valid
)

print("=========================================")

========== TUNED MODEL RELOAD ==========
Reload successful: True
Probability output valid: True


In [12]:
print()
print("================================================")
print("     STREAMSENTINEL V2 — PHASE 5 SUMMARY")
print("================================================")

print()

print(
    "ML feature count:",
    len(ML_FEATURES)
)

print(
    "Configurations tested:",
    15
)

print(
    "Best CV PR-AUC:",
    random_search.best_score_
)

print()

print("Tuned Validation Metrics")
print("------------------------")

print(
    "Accuracy:",
    tuned_accuracy
)

print(
    "Precision:",
    tuned_precision
)

print(
    "Recall:",
    tuned_recall
)

print(
    "F1:",
    tuned_f1
)

print(
    "ROC-AUC:",
    tuned_roc_auc
)

print(
    "PR-AUC:",
    tuned_pr_auc
)

print()

print(
    "Tuned model saved:",
    TUNED_MODEL_PATH.exists()
)

print(
    "Tuning results saved:",
    TUNING_RESULTS_PATH.exists()
)

print(
    "Reload test passed:",
    reload_valid
)

print()
print("PHASE 5 HYPERPARAMETER TUNING: COMPLETE")
print("================================================")


     STREAMSENTINEL V2 — PHASE 5 SUMMARY

ML feature count: 6
Configurations tested: 15
Best CV PR-AUC: 0.4654427179137611

Tuned Validation Metrics
------------------------
Accuracy: 0.9320068637091021
Precision: 0.07036326834984553
Recall: 0.8800799467021986
F1: 0.130308261405672
ROC-AUC: 0.9723830722685778
PR-AUC: 0.4777052814578071

Tuned model saved: True
Tuning results saved: True
Reload test passed: True

PHASE 5 HYPERPARAMETER TUNING: COMPLETE
